# fsq64 latent playground

This is the **quantized** sibling of `z256_latent_perturbation.ipynb`. The
tracker here does not consume a free continuous latent: its skill encoder ends
in **FSQ** (finite scalar quantization). Each of the 64 latent dimensions is
snapped independently onto a fixed lattice of 32 values, so the whole command
space is a finite grid of $32^{64}$ codes — in practice a *vocabulary* the
planner can treat as tokens.

Two facts shape every experiment below:

1. **The encoder already emits lattice values.** The oracle z is exactly on
   the grid.
2. **The tracker snaps whatever it is given back onto the grid at consume
   time** (the SONIC convention: a planner regresses the *pre-quantized*
   bounded vector, and quantization happens at the interface). So a
   perturbation only matters if it moves a dimension **past the midpoint
   between two lattice values** — smaller perturbations are erased before the
   policy ever sees them.

That second fact is the interesting difference from the continuous notebook:
the FSQ interface has a built-in dead zone of half a lattice step
($\tfrac{1}{2}\cdot\tfrac{1}{16} \approx 0.031$) around every code.

### Read every number with these caveats

- The plant is **MuJoCo**, not the Newton/PhysX simulator the policy trained
  in — a deployment-style signal, not a paper metric.
- Each rollout is **one deterministic episode**, no domain randomization, no
  pushes. Treat any difference as **preliminary** until it repeats.
- Vocabulary:
  - **z**: the 64-dim latent, lattice-valued in $\{-1, -\tfrac{15}{16},
    \dots, \tfrac{15}{16}\}$.
  - **code**: the integer index per dimension, in $[-16, 15]$
    (`pg.codes(z)`).
  - **snap**: the consume-time quantization (`pg.snap(z)`), identical to the
    tracker's (asserted in `tests/lowlevel/test_latent_playground.py`).
  - **hold / renewal / oracle z / MPJPE-L / MPJPE-G**: as in the z256
    notebook (one z per 10 ticks = 0.2 s; oracle = encoder on the reference).


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from embodied_control.lowlevel.latent import (
    LatentPlayground,
    save_grid_video,
    save_video,
    video_html,
)
from embodied_control.lowlevel.publishers.latent_perturbation import (
    ConstantLatentSource,
    SequenceLatentSource,
    TransformedLatentSource,
)
from embodied_control.lowlevel.runner import verify_bundle

np.set_printoptions(precision=3, suppress=True)

## 0. Inputs

Everything the notebook needs — the policy bundles, the reference motions,
and the G1 MuJoCo model — lives in one **playkit**. If it is not already on
disk, the cell below downloads it from the private Hugging Face dataset
`GeorgiaTech/ec-latent-playkit`, **pinned to an exact revision** so what you
run is byte-for-byte what was validated. First-time setup only:

```bash
pixi run -e latent-lab python -c "from huggingface_hub import login; login()"
```

with a Hugging Face token that can read the GeorgiaTech org (or set
`HF_TOKEN`). Already have the kit? Point `EC_LATENT_PLAYKIT` at it and the
download is skipped.

### The motions

The kit ships the 30-motion `bones_seed_language30_compositionality_v1` set
(locomotion, manipulation, idle/gesture; 14,423 frames at 50 Hz). Two of the
thirty are known to be **tracker-limited** — the oracle latent itself falls
on 4 of 5 evaluation episodes in the training simulator, so a fall there
says nothing about your perturbation: `panic_run_away_180_R_001_A423` and
`walk_big_dog_ff_225_stop_R_001_A492`.

Want different motions? The full processed catalog is public — browse
[GeorgiaTech/g1_bones_seed_sonic_129k_50hz](https://huggingface.co/datasets/GeorgiaTech/g1_bones_seed_sonic_129k_50hz)
(all 129,785 clips; `g1_bones_seed_sonic_full_manifest.json` lists every
name, with language descriptions alongside) or the curated
[GeorgiaTech/g1_bones_seed_100_50hz](https://huggingface.co/datasets/GeorgiaTech/g1_bones_seed_100_50hz),
pick clip names, and ask for a kit rebuild that includes them.


In [ ]:
PLAYKIT_REPO = "GeorgiaTech/ec-latent-playkit"  # private HF dataset
PLAYKIT_REVISION = "f0cd81ed1bd7afb821ea31d148ee19aa7fab2ab1"

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PLAYKIT = Path(os.environ.get("EC_LATENT_PLAYKIT", REPO_ROOT / "assets" / "latent_playkit")).expanduser()
if not (PLAYKIT / "playkit.json").exists():
    from huggingface_hub import snapshot_download

    print(f"playkit not found at {PLAYKIT} - downloading "
          f"{PLAYKIT_REPO}@{PLAYKIT_REVISION[:12]} (~150 MB, one time)")
    snapshot_download(
        PLAYKIT_REPO,
        repo_type="dataset",
        revision=PLAYKIT_REVISION,
        local_dir=PLAYKIT,
    )

BUNDLE = PLAYKIT / "bundles" / "fsq64_sonic_4500m"
if not BUNDLE.exists():  # v1 kits carried a single unnamed bundle
    BUNDLE = PLAYKIT / "bundle"
MODEL = PLAYKIT / "model" / "g1_29dof_rev_1_0.xml"
REFERENCE = PLAYKIT / "reference" / "root_qpos_v1"
OUTPUT = Path("runs/fsq_playground")
OUTPUT.mkdir(parents=True, exist_ok=True)

missing = [str(p) for p in (BUNDLE / "manifest.json", MODEL, REFERENCE) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "playkit is incomplete, missing:\n  " + "\n  ".join(missing)
        + f"\n\nSet EC_LATENT_PLAYKIT to the unpacked kit (currently {PLAYKIT})."
    )
print("playkit:", PLAYKIT.resolve())
print("bundle: ", BUNDLE.name)

In [ ]:
pg = LatentPlayground(BUNDLE, MODEL, REFERENCE)
command = pg.command
manifest = pg.bundle.manifest
HALF = np.asarray(command.fsq_half_levels, dtype=np.float32)
STEP = 1.0 / HALF  # lattice spacing per dimension

print(f"interface        {manifest.interface} (quantizer: {command.quantizer})")
print(f"z dimensions     {pg.z_dim}  (+ {command.phase_dim} phase)")
print(f"lattice          {int(2 * HALF[0])} levels per dimension, "
      f"step {STEP[0]:.4f}, values in [-1, {(HALF[0] - 1) / HALF[0]:.4f}]")
print(f"hold             {pg.hold_steps} ticks at {pg.control_hz} Hz = "
      f"{pg.hold_steps / pg.control_hz:.2f} s per latent")
print(f"encoder input    {command.window_steps + 1} frames x {command.state_dim} "
      f"= {(command.window_steps + 1) * command.state_dim} values "
      f"(stride {command.macro_frame_stride}, anchor {command.macro_anchor_mode})")
print(f"tracker input    {manifest.obs.total_width} values: "
      + ", ".join(f"{t.name}({t.width})" for t in manifest.obs.terms))
print(f"checkpoint       {manifest.source['checkpoint_sha256'][:16]}...")
print(f"motions          {len(pg.motion_names)}")

### Provenance gate

The bundle replays its golden trace before anything else runs — if the
TorchScript modules on disk do not reproduce the exported activations
bit-for-bit, stop here.

In [ ]:
report = verify_bundle(BUNDLE)
print(f"policy rows  {report['policy_rows']}, max abs err {report['policy_max_abs_err']:.2e}")
print(f"encoder rows {report.get('encoder_rows')}, max abs err "
      f"{report.get('encoder_max_abs_err', float('nan')):.2e}")

## 1. The code bank

Encode every motion into its sequence of oracle latents. First sanity check:
the encoder's output must already sit exactly on the lattice (`snap` is a
no-op on it).

In [ ]:
banks = {name: pg.encode_motion(name) for name in pg.motion_names}
Z_ALL = np.concatenate([bank.z for bank in banks.values()])
CODES_ALL = pg.codes(Z_ALL)
on_lattice = np.array_equal(pg.snap(Z_ALL), Z_ALL)
print(f"{len(banks)} motions -> {Z_ALL.shape[0]} latents of width {Z_ALL.shape[1]}")
print(f"encoder output already on the lattice: {on_lattice}")
assert on_lattice

pd.DataFrame(
    [
        {
            "motion": name,
            "latents": len(bank),
            "frames": int(bank.cursors[-1] + 1),
            "distinct codes": int(np.unique(pg.codes(bank.z), axis=0).shape[0]),
            "code min": int(pg.codes(bank.z).min()),
            "code max": int(pg.codes(bank.z).max()),
        }
        for name, bank in banks.items()
    ]
)

In [ ]:
# How much of the 32-level vocabulary does each dimension actually use?
levels_used = np.array([np.unique(CODES_ALL[:, d]).size for d in range(pg.z_dim)])
occupancy = np.zeros((pg.z_dim, int(2 * HALF[0])), dtype=np.int64)
for d in range(pg.z_dim):
    values, counts = np.unique(CODES_ALL[:, d] + int(HALF[0]), return_counts=True)
    occupancy[d, values] = counts

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].bar(range(pg.z_dim), np.sort(levels_used)[::-1], color="#4c72b0")
axes[0].set_title("levels used per dimension (sorted)")
axes[0].set_xlabel("dimension rank")
axes[0].set_ylabel(f"levels used (of {int(2 * HALF[0])})")
image = axes[1].imshow(occupancy.T, aspect="auto", origin="lower", cmap="magma")
axes[1].set_title("level occupancy")
axes[1].set_xlabel("dimension")
axes[1].set_ylabel("level index")
fig.colorbar(image, label="count")
plt.tight_layout()
print(f"mean levels used: {levels_used.mean():.1f} / {int(2 * HALF[0])}")
print(f"dead dimensions (1 level): {int((levels_used == 1).sum())}")

In [ ]:
# How fast does the code change? Hamming distance between consecutive renewals.
DEMO = "walk_arc_cw_start_R_slow_001_A443"
demo_motion = pg.motion(DEMO)
demo_bank = banks[DEMO]
demo_codes = pg.codes(demo_bank.z)
hamming = (np.diff(demo_codes, axis=0) != 0).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
axes[0].plot(hamming, marker=".", color="#c44e52")
axes[0].set_xlabel("renewal (one per 0.2 s)")
axes[0].set_ylabel(f"dims changed (of {pg.z_dim})")
axes[0].set_title(f"code churn - {DEMO}")
top_dims = np.argsort(demo_codes.std(axis=0))[::-1][:20]
image = axes[1].imshow(demo_codes[:, top_dims].T, aspect="auto", cmap="coolwarm")
axes[1].set_xlabel("renewal")
axes[1].set_ylabel("top-variance dimension")
axes[1].set_title("codes over time")
fig.colorbar(image, label="code")
plt.tight_layout()
print(f"mean dims changed per renewal: {hamming.mean():.1f}")

## 2. Baseline: the oracle code

Same certified path as the z256 notebook: reference window in, encoder, z
out, held for 10 ticks. The tracker's consume-time snap is a no-op here
because the oracle is already on the lattice.

In [ ]:
STEPS = 300  # 6 seconds at 50 Hz

baseline = pg.rollout(
    pg.make_reference_source(demo_motion),
    motion=demo_motion,
    max_steps=STEPS,
)
print(pg.summary(baseline, demo_motion))
video_html(save_video(baseline, OUTPUT / "baseline.mp4"))

## 3. Noise vs the lattice

The defining FSQ experiment. Add Gaussian noise to the oracle z *before* the
snap, in units of the lattice step (0.0625). The snap erases any component
that stays within half a step, so small noise should do **exactly nothing**
— unlike the continuous interface, where every sigma changes the command.
The `erased` column is the fraction of published dimensions whose consumed
value was unchanged by the noise.

In [ ]:
def noise_transform(sigma_steps, seed=0):
    """Add Gaussian noise of sigma_steps lattice steps at every renewal."""
    rng = np.random.default_rng(seed)
    scale = float(sigma_steps) * STEP

    def transform(z, renewal_index):
        return z + scale * rng.standard_normal(z.shape).astype(np.float32)

    return transform


SIGMA_STEPS = [0.0, 0.25, 0.49, 1.0, 2.0, 4.0]
noise_rollouts = {}
rows = []
for sigma in SIGMA_STEPS:
    source = TransformedLatentSource(
        pg.make_reference_source(demo_motion), noise_transform(sigma)
    )
    roll = pg.rollout(source, motion=demo_motion, max_steps=STEPS)
    consumed = pg.snap(roll.z_trace)
    oracle = pg.snap(demo_bank.z[: consumed.shape[0]])
    erased = float((pg.codes(consumed) == pg.codes(oracle)).mean())
    label = f"sigma={sigma} steps"
    noise_rollouts[label] = roll
    rows.append({"sigma_steps": sigma, "erased": round(erased, 3), **pg.summary(roll, demo_motion)})

noise_table = pd.DataFrame(rows).drop(columns=["motion"])
noise_table

In [ ]:
video_html(save_grid_video(noise_rollouts, OUTPUT / "noise_sweep.mp4", columns=3), width=900)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for label, roll in noise_rollouts.items():
    axes[0].plot(roll.base_height, label=label)
axes[0].axhline(0.4, color="k", ls="--", lw=1, label="fall threshold")
axes[0].set_xlabel("control tick")
axes[0].set_ylabel("pelvis height (m)")
axes[0].legend(fontsize=8)
axes[1].plot(noise_table["sigma_steps"], noise_table["erased"], marker="o", color="#55a868")
axes[1].set_xlabel("noise sigma (lattice steps)")
axes[1].set_ylabel("fraction erased by the snap")
axes[1].grid(alpha=0.3)
plt.tight_layout()
print("survived:", dict(zip(noise_table["sigma_steps"], noise_table["survived"])))

## 4. Random code flips

The discrete analogue of noise: pick `k` random dimensions per renewal and
move each exactly one lattice level up or down. This is the smallest
perturbation the tracker can possibly see — how many flipped dimensions does
it tolerate?

In [ ]:
def flip_transform(k, seed=0):
    rng = np.random.default_rng(seed)

    def transform(z, renewal_index):
        out = z.copy()
        dims = rng.choice(z.shape[0], size=int(k), replace=False)
        out[dims] += rng.choice([-1.0, 1.0], size=int(k)).astype(np.float32) * STEP[dims]
        return np.clip(out, -1.0, (HALF - 1.0) / HALF)

    return transform


FLIPS = [1, 4, 16, 64]
flip_rollouts = {}
rows = []
for k in FLIPS:
    source = TransformedLatentSource(
        pg.make_reference_source(demo_motion), flip_transform(k)
    )
    roll = pg.rollout(source, motion=demo_motion, max_steps=STEPS)
    flip_rollouts[f"k={k}"] = roll
    rows.append({"flipped_dims": k, **pg.summary(roll, demo_motion)})

pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(flip_rollouts, OUTPUT / "code_flips.mp4", columns=4), width=1000)

## 5. Freezing one code

Hold a single oracle code forever. With a finite vocabulary, a frozen code is
a well-defined "token" — does it encode a repeatable behaviour or only a
transition?

In [ ]:
FREEZE_AT = [0, len(demo_bank) // 3, 2 * len(demo_bank) // 3]
freeze_rollouts = {}
rows = []
for index in FREEZE_AT:
    z = demo_bank.z[index]
    roll = pg.rollout(
        ConstantLatentSource(z),
        motion=demo_motion,
        max_steps=STEPS,
        start_pose_frame=int(demo_bank.cursors[index]),
    )
    label = f"frozen@renewal {index}"
    freeze_rollouts[label] = roll
    rows.append({"frozen_at": index, **pg.summary(roll)})

pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(freeze_rollouts, OUTPUT / "frozen_codes.mp4", columns=3), width=900)

## 6. Blending two codes

Linear interpolation between two lattice points leaves the lattice; the snap
pulls every blend back to the nearest code, so a sweep over alpha moves in
**discrete jumps** — each dimension switches at its own alpha. The `dims from
B` column shows how many dimensions have already switched to motion B's code
at each alpha.

In [ ]:
MOTION_A = "walk_arc_cw_start_R_slow_001_A443"
MOTION_B = pg.motion_names[1] if pg.motion_names[0] == MOTION_A else pg.motion_names[0]
z_a = banks[MOTION_A].z[len(banks[MOTION_A]) // 2]
z_b = banks[MOTION_B].z[len(banks[MOTION_B]) // 2]

ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
blend_rollouts = {}
rows = []
for alpha in ALPHAS:
    blend = (1.0 - alpha) * z_a + alpha * z_b
    consumed = pg.snap(blend)
    from_b = int((pg.codes(consumed) == pg.codes(z_b)).sum())
    roll = pg.rollout(ConstantLatentSource(blend), max_steps=STEPS, start_pose_frame=None)
    blend_rollouts[f"alpha={alpha}"] = roll
    rows.append({"alpha": alpha, "dims from B": from_b, **pg.summary(roll)})

print(f"A = {MOTION_A}\nB = {MOTION_B}")
pd.DataFrame(rows).drop(columns=["motion"])

In [ ]:
video_html(save_grid_video(blend_rollouts, OUTPUT / "blend.mp4", columns=5), width=1100)

## 7. Your turn

`transform` receives the oracle z (already on the lattice) at every renewal
and must return a 64-dim vector; whatever you return is snapped by the
tracker before it acts. Work in codes if that is easier: `pg.codes(z)` to
integers, `codes / HALF` back to z.

In [ ]:
def my_transform(z, renewal_index):
    """Edit me: return a modified copy of the oracle latent."""
    codes = pg.codes(z).astype(np.float32)
    codes[:8] = 0.0  # example: zero the eight lowest-index codes
    return codes / HALF


custom = pg.rollout(
    TransformedLatentSource(pg.make_reference_source(demo_motion), my_transform),
    motion=demo_motion,
    max_steps=STEPS,
)
print(pg.summary(custom, demo_motion))
video_html(save_video(custom, OUTPUT / "custom.mp4"))